# RAG Evaluation Lab: Groq + Chroma + LangSmith

This is a practical replacement for the earlier mock file.

It uses an open-source RAG evaluation dataset, a local Chroma vector store, Groq for generation, and LangSmith for dataset management, evaluation, and experiment comparison.

The dataset chosen here is `philschmid/easyrag-mini-wikipedia`, which the dataset card describes as a RAG evaluation dataset with roughly 900 question/answer pairs and a separate `documents` config for retrieval. It is derived from `rag-datasets/mini_wikipedia`. 

LangSmith’s docs show the core workflow: create a dataset, run a target function on it, and compare experiments. The docs also show programmatic dataset creation and explain how to compare experiments.


## What this notebook does

1. Loads an open RAG benchmark dataset.
2. Builds a Chroma index from the document split.
3. Runs a Groq-powered RAG chain.
4. Creates a LangSmith dataset from a sample subset.
5. Evaluates a baseline chain and an improved chain.
6. Compares results side by side.
7. Computes simple RAGAS-style proxy metrics locally for quick inspection.


## 1) Install packages

```bash
pip install -U datasets python-dotenv pandas langsmith langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface langchain-chroma chromadb sentence-transformers
```

LangSmith’s quickstart notes that you need a LangSmith API key and tracing enabled to log traces and run evaluations. For non-Anthropic providers, the docs recommend the `traceable` wrapper.


In [ ]:
%pip install -qU datasets python-dotenv pandas langsmith langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface langchain-chroma chromadb sentence-transformers


## 2) Environment variables

Set these in a `.env` file or your shell:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=rag-eval-groq-demo
GROQ_API_KEY=...
GROQ_MODEL=llama-3.3-70b-versatile
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true").lower() == "true"
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "rag-eval-groq-demo")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

print("LANGSMITH_TRACING:", LANGSMITH_TRACING)
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)
print("GROQ_MODEL:", GROQ_MODEL)


## 3) Load the open dataset

The dataset has two useful configs:

- `questions` for evaluation questions and ground-truth answers
- `documents` for the source corpus

The dataset card describes it as an open RAG evaluation dataset with around 900 questions and a separate documents config for retrieval. citeturn404368view0


In [ ]:
from datasets import load_dataset

DATASET_ID = "philschmid/easyrag-mini-wikipedia"

questions_ds = load_dataset(DATASET_ID, "questions", split="full")
documents_ds = load_dataset(DATASET_ID, "documents", split="full")

print("Questions rows:", len(questions_ds))
print("Documents rows:", len(documents_ds))
print("Question columns:", questions_ds.column_names)
print("Document columns:", documents_ds.column_names)


## 4) Helpers for schema-agnostic access

The dataset format can vary by config, so the notebook uses simple helpers that look for common field names.


In [ ]:
def first_present(row, candidates, default=""):
    for key in candidates:
        if key in row and row[key] not in (None, ""):
            return row[key]
    return default

def normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())

def content_tokens(text: str) -> set[str]:
    stop = {
        'the', 'and', 'that', 'with', 'from', 'this', 'what', 'when', 'where',
        'which', 'were', 'your', 'you', 'for', 'are', 'was', 'how', 'why', 'who',
        'into', 'have', 'has', 'had', 'there', 'their', 'about', 'does', 'did',
    }
    toks = []
    for raw in normalize_text(text).split():
        token = ''.join(ch for ch in raw if ch.isalnum())
        if len(token) > 3 and token not in stop:
            toks.append(token)
    return set(toks)

def overlap_ratio(a: str, b: str) -> float:
    a_tokens = content_tokens(a)
    b_tokens = content_tokens(b)
    if not a_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens)


## 5) Sample a smaller working set

The full dataset is larger than we need for a notebook. We will use a small subset for a fast first pass, then reuse the same subset for LangSmith evaluation.


In [ ]:
DOC_LIMIT = 250
QUESTION_LIMIT = 25

doc_rows = documents_ds.select(range(min(DOC_LIMIT, len(documents_ds))))
question_rows = questions_ds.select(range(min(QUESTION_LIMIT, len(questions_ds))))

print("Working documents:", len(doc_rows))
print("Working questions:", len(question_rows))


## 6) Build LangChain documents from the corpus split


In [ ]:
from langchain_core.documents import Document

documents = []

for i, row in enumerate(doc_rows):
    text = first_present(row, ["document", "text", "content"])
    if not text:
        continue

    title = first_present(row, ["title", "page_title", "source"], default=f"doc-{i}")
    documents.append(
        Document(
            page_content=str(text),
            metadata={
                'source': DATASET_ID,
                'split': 'documents',
                'row_index': i,
                'title': str(title),
            },
        )
    )

print("Loaded documents:", len(documents))
print("Example document metadata:", documents[0].metadata if documents else {})
print("Example preview:", documents[0].page_content[:250] if documents else "")


## 7) Chunk and embed locally

We use a CPU embedding model so this stays local and inexpensive. LangChain’s Hugging Face embedding docs describe local embedding usage, and the embeddings latency docs note that local CPU models avoid network latency but still take compute time. 


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import shutil
from pathlib import Path

chunker = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

chunks = chunker.split_documents(documents)

print("Chunks:", len(chunks))

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

persist_dir = Path("./chroma_data")
if persist_dir.exists():
    shutil.rmtree(persist_dir)

vector_store = Chroma(
    collection_name="easyrag_mini_wikipedia",
    embedding_function=embeddings,
    persist_directory=str(persist_dir),
)

vector_store.add_documents(chunks)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma persistence directory:", persist_dir.resolve())


## 8) Load evaluation examples

Each question example includes an input question, a reference answer, and metadata.

LangSmith dataset examples store inputs, outputs, and optional metadata, and the docs show how to create datasets programmatically with `create_dataset` and `create_examples`. 


In [ ]:
qa_examples = []

for i, row in enumerate(question_rows):
    question = first_present(row, ["question", "query", "input"])
    answer = first_present(row, ["answer", "ground_truth", "reference", "output"])
    if not question or not answer:
        continue

    qa_examples.append(
        {
            "inputs": {"question": str(question)},
            "outputs": {"answer": str(answer)},
            "metadata": {
                'source': DATASET_ID,
                'split': 'questions',
                'row_index': i,
            },
        }
    )

print("Evaluation examples:", len(qa_examples))
print(qa_examples[0] if qa_examples else "No examples found")


## 9) Create a LangSmith dataset

The evaluation docs show that a dataset and examples are the core inputs to `Client.evaluate`. The dataset management docs also show that examples can be added programmatically using the Python SDK.


In [ ]:
from langsmith import Client

client = Client()
dataset_name = "easyrag-mini-wikipedia-rag-demo"

if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description='RAG evaluation sample built from easyrag-mini-wikipedia',
    )
    client.create_examples(dataset_id=dataset.id, examples=qa_examples)
    print("Created dataset:", dataset_name)
else:
    dataset = None
    print("Dataset already exists:", dataset_name)


## 10) Set up the Groq RAG chain

We will build two versions:

- Baseline RAG: retrieve and answer directly
- Improved RAG: rewrite the question before retrieval

LangSmith’s quickstart notes that for non-Anthropic providers you should use the `traceable` wrapper so the call is traced. 


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
)

answer_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You answer using only the provided context. If the context is insufficient, say you do not know.'),
    ('user', 'Question: {question}\n\nContext: {context}\n\nAnswer in 3 sentences or fewer.'),
])

rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Rewrite the question into a short standalone search query. Return only the rewritten query.'),
    ('user', '{question}'),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

print("Groq model ready.")


## 11) Utility: build context from retrieved chunks


In [ ]:
def make_context(docs, max_chars_per_doc=1200):
    parts = []
    for d in docs:
        text = d.page_content.strip()
        if len(text) > max_chars_per_doc:
            text = text[:max_chars_per_doc].rsplit(' ', 1)[0] + '...'
        title = d.metadata.get('title', 'unknown')
        parts.append(f'[{title}] {text}')
    return '\n\n'.join(parts)


## 12) Baseline RAG target

This is the first version we will evaluate. It retrieves with the raw question and answers from the retrieved context.

The LangSmith RAG evaluation tutorial uses a target function plus evaluators for answer quality and retrieval quality, and the docs show the same overall experiment pattern.


In [ ]:
@traceable(name='rag_baseline')
def rag_baseline(inputs: dict) -> dict:
    question = inputs['question']
    docs = retriever.invoke(question)
    context = make_context(docs)
    result = answer_prompt.format_messages(question=question, context=context)
    response = llm.invoke(result)

    return {
        'answer': response.content,
        'documents': [
            {'page_content': d.page_content, 'metadata': d.metadata}
            for d in docs
        ],
        'context': context,
    }


## 13) Improved RAG target

This version rewrites the question before retrieval. That mirrors the rewrite step described in the LangGraph agentic RAG guide and keeps the change small enough to compare cleanly.


In [ ]:
@traceable(name='rag_rewrite')
def rag_rewrite(inputs: dict) -> dict:
    question = inputs['question']
    rewritten = rewrite_chain.invoke({'question': question})
    docs = retriever.invoke(rewritten)
    context = make_context(docs)
    result = answer_prompt.format_messages(question=question, context=context)
    response = llm.invoke(result)

    return {
        'answer': response.content,
        'documents': [
            {'page_content': d.page_content, 'metadata': d.metadata}
            for d in docs
        ],
        'context': context,
        'rewritten_question': rewritten,
    }


## 14) Local proxy metrics for faithfulness, context precision, and recall

These are simple, transparent proxies so you can inspect behavior quickly.

They are not the full RAGAS library, but they are useful for a first notebook pass:
- faithfulness proxy: answer supported by retrieved context
- context precision proxy: retrieved context aligned with the question
- recall proxy: reference answer supported by retrieved context


In [ ]:
def faithfulness_proxy(answer: str, context: str) -> float:
    return overlap_ratio(answer, context)

def context_precision_proxy(question: str, docs) -> float:
    if not docs:
        return 0.0
    question_tokens = content_tokens(question)
    if not question_tokens:
        return 0.0
    scores = []
    for d in docs:
        scores.append(len(question_tokens & content_tokens(d['page_content'])) / len(question_tokens))
    return sum(scores) / len(scores)

def recall_proxy(reference_answer: str, context: str) -> float:
    return overlap_ratio(reference_answer, context)


## 15) Define LangSmith evaluators

The LangSmith RAG tutorial uses evaluators for correctness, groundedness, relevance, and retrieval relevance. Here we keep the same evaluation shape, but use small local proxy evaluators so the notebook stays easy to understand and fast to run. 


In [ ]:
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    ref = normalize_text(reference_outputs.get('answer', ''))
    ans = normalize_text(outputs.get('answer', ''))
    if not ref or not ans:
        return False
    return overlap_ratio(ref, ans) >= 0.35

def groundedness(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    docs = outputs.get('documents', [])
    context = '\n'.join(d.get('page_content', '') for d in docs)
    return faithfulness_proxy(outputs.get('answer', ''), context) >= 0.20

def retrieval_relevance(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    docs = outputs.get('documents', [])
    return context_precision_proxy(inputs.get('question', ''), docs) >= 0.20


## 16) Run the LangSmith evaluation

LangSmith’s evaluation docs show `client.evaluate(target, data=dataset_name, evaluators=[...], experiment_prefix=...)` as the main way to run an experiment. The docs also note that you can run locally without uploading results by setting `upload_results=False`. 


In [ ]:
UPLOAD_RESULTS = bool(LANGSMITH_API_KEY)

baseline_experiment = client.evaluate(
    rag_baseline,
    data=dataset_name,
    evaluators=[correctness, groundedness, retrieval_relevance],
    experiment_prefix='baseline-groq-chroma',
    metadata={'version': 'baseline'},
    upload_results=UPLOAD_RESULTS,
)

rewrite_experiment = client.evaluate(
    rag_rewrite,
    data=dataset_name,
    evaluators=[correctness, groundedness, retrieval_relevance],
    experiment_prefix='rewrite-groq-chroma',
    metadata={'version': 'rewrite'},
    upload_results=UPLOAD_RESULTS,
)

print("Evaluation launched.")


## 17) Compare the two chain versions

LangSmith’s comparison docs are intended for exactly this kind of iteration: compare experiments side by side, identify regressions, and inspect which version is better on the dataset.


In [ ]:
import pandas as pd

def experiment_to_df(experiment):
    if hasattr(experiment, 'to_pandas'):
        return experiment.to_pandas()
    try:
        return pd.DataFrame(list(experiment))
    except Exception:
        return pd.DataFrame()

baseline_df = experiment_to_df(baseline_experiment)
rewrite_df = experiment_to_df(rewrite_experiment)

print("Baseline results shape:", baseline_df.shape)
print("Rewrite results shape:", rewrite_df.shape)

baseline_df.head()


In [ ]:
def local_run(target_fn, examples, label):
    rows = []
    for ex in examples:
        inputs = ex['inputs']
        reference = ex['outputs']
        outputs = target_fn(inputs)
        docs = outputs.get('documents', [])
        context = outputs.get('context', '')
        rows.append({
            'experiment': label,
            'question': inputs['question'],
            'reference_answer': reference['answer'],
            'answer': outputs['answer'],
            'correctness_proxy': correctness(inputs, outputs, reference),
            'faithfulness_proxy': faithfulness_proxy(outputs['answer'], context),
            'context_precision_proxy': context_precision_proxy(inputs['question'], docs),
            'recall_proxy': recall_proxy(reference['answer'], context),
            'rewritten_question': outputs.get('rewritten_question', ''),
        })
    return rows

sample_eval = qa_examples[:10]
baseline_local = local_run(rag_baseline, sample_eval, 'baseline')
rewrite_local = local_run(rag_rewrite, sample_eval, 'rewrite')
local_compare = pd.DataFrame(baseline_local + rewrite_local)

local_compare


## 18) A short summary table

This gives you a simple way to spot which version is doing better.


In [ ]:
summary = (
    local_compare
    .groupby('experiment')[[
        'correctness_proxy',
        'faithfulness_proxy',
        'context_precision_proxy',
        'recall_proxy',
    ]]
    .mean()
    .reset_index()
)

summary


## 19) What to look at next

Useful next steps after this notebook:

- inspect failed examples in LangSmith
- add better query rewriting
- add contextual compression
- try a reranker
- expand the LangSmith dataset
- replace proxy scores with a stronger judge when needed

LangSmith supports dataset versioning, filtering, and comparing experiments over time, which makes that iterative loop practical. 


## Key takeaways

- This notebook now uses a real open-source RAG evaluation dataset.
- Groq handles generation, Chroma stores local embeddings, and LangSmith manages the evaluation workflow.
- You get two chain versions to compare instead of a single mock function.
- The local proxy metrics make faithfulness, context precision, and recall easy to inspect right away.
- LangSmith then gives you persistent datasets and experiment comparisons for the same examples. 


## References

- RAG evaluation dataset: https://huggingface.co/datasets/philschmid/easyrag-mini-wikipedia
- Chroma local persistence: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
- Evaluation quickstart: https://docs.langchain.com/langsmith/evaluation-quickstart
- Evaluate a RAG application: https://docs.langchain.com/langsmith/evaluate-rag-tutorial
- Compare experiment results: https://docs.langchain.com/langsmith/compare-experiment-results
- Manage datasets programmatically: https://docs.langchain.com/langsmith/manage-datasets-programmatically
- Tracing quickstart: https://docs.langchain.com/langsmith/observability-quickstart
- Create an API key: https://docs.langchain.com/langsmith/create-account-api-key
